# 🧬 MiroFish-BioReviewer
### AI-Assisted Review of Systems Biology Grant Pre-Proposals

This notebook runs MiroFish-BioReviewer in Google Colab. It will:
1. Prompt you for API keys (stored only in session memory)
2. Accept your grant pre-proposal PDF upload
3. Build a knowledge graph from the proposal
4. Run a swarm simulation of biological entities from the proposal
5. Convene a 3-agent expert reviewer panel
6. Generate a structured review report

**Required API keys:** LLM provider key + ZEP Cloud key  
**Estimated runtime:** 5–15 minutes depending on model and simulation rounds  
**Default simulation length:** 40 rounds

In [ ]:
# ── Cell 2: Environment Setup ──────────────────────────────────────────────
import subprocess, sys, os

print("📦 Installing MiroFish-BioReviewer dependencies...")

# Clone repo
if not os.path.exists("MiroFish-BioReviewer"):
    subprocess.run([
        "git", "clone", "--depth=1",
        "https://github.com/kouroshSA/MiroFish-BioReviewer.git"
    ], check=True)

os.chdir("MiroFish-BioReviewer")
sys.path.insert(0, os.path.abspath("backend"))

# Install Python backend dependencies
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                "backend/requirements.txt"], check=True)

print("✅ Setup complete.")

In [ ]:
# ── Cell 3: API Key Configuration ──────────────────────────────────────────
# Keys are entered interactively and stored ONLY in session memory.
# They are never written to disk, never logged, and lost when the session ends.

import getpass
import os

print("🔑 API Key Configuration")
print("=" * 50)
print("Keys are stored in session memory only (getpass).")
print("They will NOT be saved to disk or appear in notebook output.\n")

# LLM Provider selection
print("Choose your LLM provider:")
print("  1. OpenAI (gpt-4o, gpt-4o-mini, gpt-4.1)")
print("  2. Anthropic Claude (via LiteLLM proxy — see README)")
print("  3. Google Gemini (gemini-2.5-flash)")
print("  4. DeepSeek (deepseek-chat)")
print("  5. Custom OpenAI-compatible endpoint")

provider_choice = input("\nEnter choice (1-5): ").strip()

PROVIDER_DEFAULTS = {
    "1": ("https://api.openai.com/v1", "gpt-4o-mini"),
    "2": ("http://localhost:8000/v1", "claude-sonnet-4-20250514"),
    "3": ("https://generativelanguage.googleapis.com/v1beta/openai", "gemini-2.5-flash"),
    "4": ("https://api.deepseek.com/v1", "deepseek-chat"),
    "5": (None, None),
}

default_base_url, default_model = PROVIDER_DEFAULTS.get(provider_choice, (None, None))

if provider_choice == "5":
    llm_base_url = input("Enter your custom base URL (e.g. http://localhost:11434/v1): ").strip()
    llm_model = input("Enter your model name: ").strip()
else:
    llm_base_url = default_base_url
    llm_model = input(f"Model name [{default_model}]: ").strip() or default_model

llm_api_key = getpass.getpass("\nEnter your LLM API key (hidden): ")

print("\nNow enter your ZEP Cloud API key.")
print("Get one free at: https://app.getzep.com/")
zep_api_key = getpass.getpass("ZEP API key (hidden): ")

# Set environment variables for the backend
os.environ["LLM_API_KEY"] = llm_api_key
os.environ["LLM_BASE_URL"] = llm_base_url
os.environ["LLM_MODEL_NAME"] = llm_model
os.environ["ZEP_API_KEY"] = zep_api_key
os.environ["SIMULATION_MODE"] = "grant_review"
os.environ["REVIEWER_PANEL_ENABLED"] = "true"

# Optional: stronger model for Discussion/Big Picture sections
use_strong_model = input(
    "\nUse a stronger model for final report synthesis? "
    "Enter model name or press Enter to use same model: "
).strip()
if use_strong_model:
    os.environ["DISCUSSION_LLM_MODEL_NAME"] = use_strong_model

print("\n✅ Configuration complete. Keys stored in session memory only.")
print(f"   Provider: {llm_base_url}")
print(f"   Model: {llm_model}")
if zep_api_key:
    print(f"   ZEP key: {'*' * (len(zep_api_key) - 4) + zep_api_key[-4:]}")

In [ ]:
# ── Cell 4: Upload Grant Pre-Proposal ──────────────────────────────────────
from google.colab import files
import os

print("📄 Upload your grant pre-proposal PDF (2–3 pages).")
print("Supported formats: PDF, Markdown (.md), Plain text (.txt)\n")

uploaded = files.upload()

if not uploaded:
    raise ValueError("No file uploaded. Please re-run this cell and upload a file.")

uploaded_filename = list(uploaded.keys())[0]
uploaded_path = os.path.abspath(uploaded_filename)
print(f"\n✅ Uploaded: {uploaded_filename} ({len(uploaded[uploaded_filename]):,} bytes)")

# Store path for downstream cells
os.environ["PROPOSAL_PATH"] = uploaded_path

In [ ]:
# ── Cell 5: Simulation Parameters ──────────────────────────────────────────
# Default rounds = 40 (Colab-specific default for MiroFish-BioReviewer)

max_rounds_input = input(
    "Max simulation rounds [default: 40, recommended: 20–60]: "
).strip()
max_rounds = int(max_rounds_input) if max_rounds_input.isdigit() else 40

simulation_request = input(
    "\nDescribe your review focus (or press Enter for default):\n"
    "Default: 'Review this systems biology grant pre-proposal. Evaluate scientific "
    "significance, innovation, mechanistic rigor, quantitative design, and feasibility.'\n> "
).strip()

if not simulation_request:
    simulation_request = (
        "Review this systems biology grant pre-proposal. "
        "Simulate how the key biological tools, systems, and targets described in "
        "the proposal would react to and debate the proposed research. "
        "Evaluate scientific significance, innovation, mechanistic rigor, "
        "quantitative and statistical design, and feasibility. "
        "Focus on entities described in the proposal — do NOT create agents for "
        "researchers, institutions, or journals."
    )

os.environ["MAX_ROUNDS"] = str(max_rounds)
os.environ["SIMULATION_REQUEST"] = simulation_request

print(f"\n✅ Parameters set: {max_rounds} rounds (default 40 in Colab)")

In [ ]:
# ── Cell 6: Run MiroFish-BioReviewer Pipeline ──────────────────────────────
# This cell runs the full pipeline:
# 1. Graph construction from the uploaded proposal
# 2. Swarm simulation (grant_review mode)
# 3. Reviewer panel (The Mechanist, The Visionary, The Realist)
# 4. Report generation

import subprocess, sys, os

output_dir = "colab_output"
os.makedirs(output_dir, exist_ok=True)

cmd = [
    sys.executable, "backend/scripts/run_review_pipeline.py",
    "--proposal",   os.environ["PROPOSAL_PATH"],
    "--request",    os.environ["SIMULATION_REQUEST"],
    "--max-rounds", os.environ["MAX_ROUNDS"],
    "--output-dir", output_dir,
    "--mode",       "grant_review",
]
print("$ " + " ".join(cmd))
result = subprocess.run(cmd)

if result.returncode != 0:
    raise RuntimeError(f"Pipeline exited with status {result.returncode}")

os.environ["OUTPUT_DIR"] = output_dir
print("\n✅ Pipeline complete.")

In [ ]:
# ── Cell 7: Display and Download Report ────────────────────────────────────
import os, json
from IPython.display import Markdown, display
from google.colab import files

output_dir = os.environ.get("OUTPUT_DIR", "colab_output")

# Find the report file
report_candidates = ["full_report.md", "report.md", "LLM_Refined_Report.md"]
report_path = None
for candidate in report_candidates:
    p = os.path.join(output_dir, candidate)
    if os.path.exists(p):
        report_path = p
        break

if report_path:
    with open(report_path, "r", encoding="utf-8") as f:
        report_text = f.read()
    print(f"📋 Report found: {report_path}\n")
    display(Markdown(report_text))

    # Download
    download_confirm = input("\nDownload report as Markdown? (y/n): ").strip().lower()
    if download_confirm == "y":
        files.download(report_path)
else:
    print("⚠️  Report file not found. Check output directory:")
    print(os.listdir(output_dir))

# Also display reviewer panel if available
panel_path = os.path.join(output_dir, "reviewer_panel.json")
if os.path.exists(panel_path):
    with open(panel_path, "r", encoding="utf-8") as f:
        panel = json.load(f)

    print("\n\n📊 Reviewer Panel Raw Data:")
    for review in panel.get("reviews", []):
        print(f"\n{'='*50}")
        print(f"Reviewer: {review.get('reviewer', 'Unknown')}")
        print(f"Recommendation: {review.get('recommendation', 'N/A')}")
        scores = review.get("dimension_scores", {})
        for dim, score in scores.items():
            print(f"  {dim}: {score}/10")